# Automatización Empírica de Hiperparámetros (Grid Search)

Este script representa el rigor científico de tu investigación. No elegimos 128 neuronas al azar, sino que programamos este torneo para enfrentar múltiples arquitecturas bajo el mismo dataset y que las matemáticas revelen la configuración con el menor margen de error.

In [1]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(888)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import itertools
from sklearn.preprocessing import StandardScaler

from networks import StockDataset, MasterStockTransformer

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Torneo corriendo en: {device}")

Torneo corriendo en: mps


### 1. Carga de Datos

In [2]:
df = pd.read_csv('processed_market_data.csv', index_col=0, parse_dates=True)

target_idx = df.columns.get_loc('JPM_Log_Ret')
ma5_idx = df.columns.get_loc('JPM_Target_MA5')
close_idx = df.columns.get_loc('JPM_Close')
context_indices = [i for i in range(len(df.columns)) if i not in [target_idx, ma5_idx, close_idx]]

train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:train_size+val_size]

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)

### 2. Definición del Espacio de Búsqueda (Grid)

In [3]:
hyper_grid = {
    'd_model': [64, 128],       # Modelo ligero vs pesado
    'num_layers': [1, 2],       # Modelo superficial vs profundo
    'epochs': [30, 50],         # Entrenamientos rápidos vs largos (¡A petición tuya!)
    'dropout': [0.1, 0.2]       # Diferentes niveles de regularización (para evitar overfitting)
}

# Generamos todas las combinaciones posibles matemáticamente (2x2x2x2 = 16 experimentos)
keys, values = zip(*hyper_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Se han programado {len(experiments)} experimentos/redes neuronales a competir.")

Se han programado 16 experimentos/redes neuronales a competir.


### 3. Loop del Torneo de Optimización

In [4]:
results = []
seq_length = 20 # Mantenemos la memoria en 20 días para igualar condiciones
batch_size = 64

train_dataset = StockDataset(train_scaled, target_idx, context_indices, seq_length)
val_dataset = StockDataset(val_scaled, target_idx, context_indices, seq_length)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

criterion = nn.MSELoss()

for i, params in enumerate(experiments):
    print(f"\n--- Iniciando Experimento {i+1}/{len(experiments)} ---")
    print(f"Parámetros: {params}")
    
    model = MasterStockTransformer(target_features=1, 
                                   context_features=len(context_indices), 
                                   d_model=params['d_model'], 
                                   nhead=4, 
                                   num_layers=params['num_layers'],
                                   dropout=params['dropout']).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=0.0005)
    
    best_val_loss = float('inf')
    
    for epoch in range(params['epochs']):
        model.train()
        for x_tgt, x_ctx, y_true in train_loader:
            x_tgt, x_ctx, y_true = x_tgt.to(device), x_ctx.to(device), y_true.to(device)
            optimizer.zero_grad()
            predictions, _ = model(x_tgt, x_ctx)
            loss = criterion(predictions, y_true)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
        # Calcular Validación
        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for x_tgt, x_ctx, y_true in val_loader:
                x_tgt, x_ctx, y_true = x_tgt.to(device), x_ctx.to(device), y_true.to(device)
                preds, _ = model(x_tgt, x_ctx)
                val_loss = criterion(preds, y_true)
                epoch_val_loss += val_loss.item()
                
        avg_val_loss = epoch_val_loss / len(val_loader)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            
    print(f"=> Mejor MSE de Validación alcanzado: {best_val_loss:.6f}")
    results.append({
        'd_model': params['d_model'],
        'num_layers': params['num_layers'],
        'epochs': params['epochs'],
        'dropout': params['dropout'],
        'Validation MSE': best_val_loss
    })

print("\n=== ¡Torneo Finalizado! ===")


--- Iniciando Experimento 1/16 ---
Parámetros: {'d_model': 64, 'num_layers': 1, 'epochs': 30, 'dropout': 0.1}
=> Mejor MSE de Validación alcanzado: 0.716047

--- Iniciando Experimento 2/16 ---
Parámetros: {'d_model': 64, 'num_layers': 1, 'epochs': 30, 'dropout': 0.2}
=> Mejor MSE de Validación alcanzado: 0.713894

--- Iniciando Experimento 3/16 ---
Parámetros: {'d_model': 64, 'num_layers': 1, 'epochs': 50, 'dropout': 0.1}
=> Mejor MSE de Validación alcanzado: 0.716314

--- Iniciando Experimento 4/16 ---
Parámetros: {'d_model': 64, 'num_layers': 1, 'epochs': 50, 'dropout': 0.2}
=> Mejor MSE de Validación alcanzado: 0.715422

--- Iniciando Experimento 5/16 ---
Parámetros: {'d_model': 64, 'num_layers': 2, 'epochs': 30, 'dropout': 0.1}
=> Mejor MSE de Validación alcanzado: 0.711738

--- Iniciando Experimento 6/16 ---
Parámetros: {'d_model': 64, 'num_layers': 2, 'epochs': 30, 'dropout': 0.2}
=> Mejor MSE de Validación alcanzado: 0.710116

--- Iniciando Experimento 7/16 ---
Parámetros: {'d_

### 4. Tabla de Resultados (La Prueba Científica)

In [5]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Validation MSE', ascending=True).reset_index(drop=True)

print("=== Leaderboard de Hiperparámetros ===")
display(results_df)

best_params = results_df.iloc[0]
print(f"\n¡LA MEJOR COMBINACIÓN FUE: {best_params.to_dict()}!")
print("Puedes usar esta tabla en tu documento final para probar por qué elegiste esos parámetros.")

=== Leaderboard de Hiperparámetros ===


,d_model,num_layers,epochs,dropout,Validation MSE
0,64,2,50,0.1,0.703925
1,128,1,30,0.1,0.709687
2,128,1,30,0.2,0.709867
3,64,2,30,0.2,0.710116
4,64,2,50,0.2,0.710711
5,64,2,30,0.1,0.711738
6,128,1,50,0.2,0.713632
7,64,1,30,0.2,0.713894
8,128,2,30,0.1,0.714444
9,64,1,50,0.2,0.715422



¡LA MEJOR COMBINACIÓN FUE: {'d_model': 64.0, 'num_layers': 2.0, 'epochs': 50.0, 'dropout': 0.1, 'Validation MSE': 0.7039251327514648}!
Puedes usar esta tabla en tu documento final para probar por qué elegiste esos parámetros.
